### Assess Recovery

We fit a bunch of injected events, or at least those that passed our event and anomaly detection pipeline. Let's assess how well we did in 1) detection and 2) classification. 

#### Detection Recovery

In [26]:
# system imports
import os
import time
import sys
import glob
from tqdm import tqdm
#from pathlib import Path

# data access imports
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# display imports
from IPython.display import HTML

# multiprocessing imports
import multiprocessing as mp
from multiprocessing.pool import ThreadPool

# data challenge imports
#import MulensModel
import emcee
#from pyLIMA.pyLIMASS import SourceLensProbabilities
import RTModel

# data analysis/visualization imports
import numpy as np
import matplotlib.pyplot as plt

#### IMPORTANT: Who are you? 

In [2]:
event_data_me = 'event_data_chris/' # please change 'chris' to your name, otherwise you will overwrite my work

#### Mise en place

In [3]:
TIER = "Beginner" # change to Experienced or ML if you want to not be a baby
root_dir = '/home/mldc-ipacpeople-admin/teams/mldc-ipacpeople/'
events_dir = root_dir + event_data_me

In [4]:
def read_nature_txt(filename, n_chisquares):
    """
    Read Nature.txt if it's not empty and grab the top n_chisquares models. 
    """
    found = False
    n = 0
    chisqs = []
    model_files = []
    with open(filename, "r") as file:
        for line in file:
            # don't do anything until the chisquare line, which is where FinalModels are listed
            if 'chisquare' in line:
                found = True 
                continue # skip to next line, our search for the chisquare line is done
            
            # only then do we start processing lines
            if found == True:
                #print(line.strip())  # strip() removes trailing newlines

                # grab best N chisquares
                if n < n_chisquares:
                    chisq = line.split()[0]
                    model_file = line.split()[1]
                    chisqs.append(chisq)
                    model_files.append(model_file)
                else: 
                    continue
    
                n += 1
    file.close() 

    return chisqs, model_files

In [31]:
if TIER == "Beginner":
    assess_range = np.arange(1,189)

n_chisquares = 1 # how many of the best chisquares to take
s_list, q_list, u0_list, alpha_list, rho_list, tE_list, t0_list = [], [], [], [], [], [], []
event_list, chisq_list = [], []
for i in tqdm(assess_range):
    event_dir = events_dir + 'event_000' + "{:03d}/".format(i)
    event_final_models = glob.glob(event_dir+'FinalModels/*')

    # if there are no FinalModels (ie., if either there was no detection or there was no 
    if len(event_final_models)==0:
        continue
    else:
        chisqs, model_filenames = read_nature_txt(event_dir+'Nature.txt', n_chisquares)

    # read N best model_filenames to get inferred parameters
    for j in range(n_chisquares):
        chisq_temp = chisqs[j]
        model_filename_temp = model_filenames[j]
        with open(event_dir + 'FinalModels/' + model_filename_temp, "r") as model_file:
            # look at only the first two lines. The rest is the covariance matrix, which we're not interested in for now.
            lines = model_file.readlines()
            line1 = lines[0]
            line2 = lines[1]
    
            # FinalModels/ files have variable columns depending on the model archetype. Account for this.
            if model_filename_temp[:2] == 'LS':
                s,q,u0,alpha,rho,tE,t0,back_flux,source_flux,anomaly_t0,anomaly_y1,anomaly_y2,anomaly_Delta_chisq,chisq = line1.split()
                s_err,q_err,u0_err,alpha_err,rho_err,tE_err,t0_err,anomaly_t0_err,anomaly_y1_err,anomaly_y2_err,anomaly_Delta_chisq_err,unk5_err,unk6_err = line2.split()
        event_list.append(int(i))
        chisq_list.append(float(chisq))
        s_list.append(float(s))
        q_list.append(float(q))
        u0_list.append(float(u0))
        alpha_list.append(float(alpha))
        rho_list.append(float(rho))
        tE_list.append(float(tE))
        t0_list.append(float(t0))
        
        model_file.close() 

100%|██████████| 188/188 [00:00<00:00, 382.17it/s]


In [37]:
analyze_df = pd.DataFrame({'event':event_list,'chisq':chisq_list,'s':s_list,'q':q_list,'u0':u0_list,'alpha':alpha_list,'rho':rho_list,
                        'tE':tE_list,'t0':t0_list})

In [38]:
analyze_df

,event,chisq,s,q,u0,alpha,rho,tE,t0
0,2,6949.109170,0.767449,0.000141,0.139412,4.972064,0.006046,5.245622,12751.922817
1,3,6949.109170,0.767449,0.000141,0.139412,4.972064,0.006046,5.245622,12751.922817
2,4,6949.109170,0.767449,0.000141,0.139412,4.972064,0.006046,5.245622,12751.922817
3,5,7606.031290,0.901214,0.000843,0.203245,4.598937,0.000617,10.414131,11854.946206
4,6,7442.816461,0.801030,0.000107,0.417570,4.506630,0.003026,15.593170,13116.116040
...,...,...,...,...,...,...,...,...,...
135,184,7028.867314,0.914296,0.000012,0.061073,4.015509,0.001608,16.203776,12745.319237
136,185,6962.637400,1.064998,0.000008,0.004512,4.014410,0.000356,34.024217,11494.847636
137,186,6962.637400,1.064998,0.000008,0.004512,4.014410,0.000356,34.024217,11494.847636
138,187,6962.637400,1.064998,0.000008,0.004512,4.014410,0.000356,34.024217,11494.847636
